<a href="https://colab.research.google.com/github/trongtin142/cofog-panel/blob/main/cofog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COFOG-Panel: Harmonizing IMF GFS COFOG Data into Reproducible Panel Datasets

This notebook installs COFOG-Panel and runs the pipeline. Choose the function, transformation and method in the form (cell 3), then run cell 4 for the full pipeline or cells 5a–5d for the individual stages.

The IMF data are subject to the IMF terms of use (see `DATA_LICENSE.md` in the repository).

In [ ]:
# @title 1. Setup Environment and Install the Package

# Use a tagged release for reproducible results (replace v1.0.2 with the version cited in your work)
!git clone --depth 1 --branch v1.0.2 https://github.com/trongtin142/cofog-panel.git || git clone --depth 1 https://github.com/trongtin142/cofog-panel.git
%cd cofog-panel

# Install the package (and its dependencies) to get the `cofog-panel` command
!pip install -q -e .
!cofog-panel --version

In [ ]:
# @title 2. Prepare Data
# @markdown Place the input files in ./data/:
# @markdown - gfs_raw_data.xlsx: IMF GFS COFOG extract (Excel download from https://data.imf.org/en/datasets/IMF.STA:GFS_COFOG)
# @markdown   (not included in the repository: download it and upload it to ./data/ with the Colab Files panel)
# @markdown - country_codes.xlsx: country lookup (included in the repository)

import os

os.makedirs('data', exist_ok=True)
os.makedirs('output', exist_ok=True)
for f in ['data/gfs_raw_data.xlsx', 'data/country_codes.xlsx']:
    print(f, 'found' if os.path.exists(f) else 'MISSING')

In [ ]:
# @title 3. Input Form
# @markdown * **COFOG**: a division or group code (e.g. 05 Environmental protection, 07 Health, 07.3 Hospital services).
# @markdown * **Transformation**: how values are expressed. Shares of total outlays are never summed across subsectors.
# @markdown * **Method**: 'General government' applies the three-tier rule; the other options select a single sector.
# @markdown * **Data column**: name of the output column (traceability columns use the default SOURCE_* names).

import ipywidgets as widgets
from IPython.display import display

COFOG_DIVISIONS = [('01 General public services', '01'), ('02 Defence', '02'), ('03 Public order and safety', '03'),
                   ('04 Economic affairs', '04'), ('05 Environmental protection', '05'),
                   ('06 Housing and community amenities', '06'), ('07 Health', '07'),
                   ('08 Recreation, culture and religion', '08'), ('09 Education', '09'), ('10 Social protection', '10')]
dropdown_cofog = widgets.Dropdown(options=COFOG_DIVISIONS, value='05', description='COFOG:')
dropdown_transformation = widgets.Dropdown(options=['Percent of GDP', 'Domestic currency', 'Percent of total outlays'],
                                           description='Transformation:')
dropdown_sector = widgets.Dropdown(options=['General government', 'Central government including social security',
                                            'Central government excluding social security', 'Budgetary central government',
                                            'Extrabudgetary central government', 'Social security funds',
                                            'State Government', 'Local Government'], description='Method:')
input_data_col = widgets.Text(value='DATA_ENV', description='Data column:')
for w in (dropdown_cofog, dropdown_transformation, dropdown_sector, input_data_col):
    display(w)

In [ ]:
print(dropdown_cofog.value, dropdown_transformation.value, dropdown_sector.value, input_data_col.value)

In [ ]:
# @title 4. Orchestrator: Run Full Pipeline
# @markdown Runs stages 1-5 with the values chosen in the form.

!cofog-panel run \
    --source-file ./data/gfs_raw_data.xlsx \
    --lookup-file ./data/country_codes.xlsx \
    --cofog "{dropdown_cofog.value}" \
    --filter-type "{dropdown_transformation.value}" \
    --sector "{dropdown_sector.value}" \
    --data-col "{input_data_col.value}"

### Individual Stages (Manual Execution)

In [ ]:
# @title 5a. Stages 1 & 2: Validation

!cofog-panel check-format --source-file ./data/gfs_raw_data.xlsx
!cofog-panel check-country-format --lookup-file ./data/country_codes.xlsx

In [ ]:
# @title 5b. Stage 3: Seeding the Master Panel
!cofog-panel seed-master --lookup-file ./data/country_codes.xlsx --output-dir ./output

In [ ]:
# @title 5c. Stage 4: ETL and Data Splitting
!cofog-panel split --source-file ./data/gfs_raw_data.xlsx --cofog "{dropdown_cofog.value}" --filter-type "{dropdown_transformation.value}" --output-dir ./intermediate_splits

In [ ]:
# @title 5d. Stage 5: Harmonisation and Aggregation
!cofog-panel aggregate --master-file ./output/COFOG_MASTER_SCHEMA.xlsx --folder-path ./intermediate_splits --data-col "{input_data_col.value}" --sector "{dropdown_sector.value}"